<a href="https://colab.research.google.com/github/rohan-sarker-iem/Oracle/blob/main/Clustering_Fixed_Rotating_Axis_Ellipse_Data_Points.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Bug-Free Python Code for Clustering Fixed and Angular Elliptical Data Points

Features:
1. Generates synthetic elliptical clusters
2. Supports:
   - Fixed ellipses
   - Rotated/angular ellipses
3. Uses Gaussian Mixture Model (GMM)
4. Draws ellipse boundaries
5. Fully stable and tested

Required Libraries:
pip install numpy matplotlib scikit-learn
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.mixture import GaussianMixture


# ============================================================
# Generate Elliptical Data
# ============================================================

def generate_ellipse_points(center,
                            major_axis,
                            minor_axis,
                            angle_deg,
                            n_points,
                            noise=0.3):
    """
    Generate points inside a rotated ellipse.

    Parameters:
    -----------
    center : tuple
        (x, y) center of ellipse

    major_axis : float
        Length of major axis

    minor_axis : float
        Length of minor axis

    angle_deg : float
        Rotation angle in degrees

    n_points : int
        Number of points

    noise : float
        Gaussian noise level
    """

    angle_rad = np.radians(angle_deg)

    # Random points
    theta = np.random.uniform(0, 2 * np.pi, n_points)
    r = np.sqrt(np.random.uniform(0, 1, n_points))

    x = major_axis * r * np.cos(theta)
    y = minor_axis * r * np.sin(theta)

    # Rotation matrix
    R = np.array([
        [np.cos(angle_rad), -np.sin(angle_rad)],
        [np.sin(angle_rad),  np.cos(angle_rad)]
    ])

    points = np.vstack((x, y))
    rotated = R @ points

    rotated[0, :] += center[0]
    rotated[1, :] += center[1]

    # Add noise
    rotated += np.random.normal(0, noise, rotated.shape)

    return rotated.T


# ============================================================
# Create Dataset
# ============================================================

np.random.seed(42)

cluster1 = generate_ellipse_points(
    center=(2, 3),
    major_axis=5,
    minor_axis=1.5,
    angle_deg=0,          # Fixed ellipse
    n_points=400
)

cluster2 = generate_ellipse_points(
    center=(12, 10),
    major_axis=4,
    minor_axis=1,
    angle_deg=45,         # Angular ellipse
    n_points=400
)

cluster3 = generate_ellipse_points(
    center=(20, 5),
    major_axis=6,
    minor_axis=2,
    angle_deg=75,         # Angular ellipse
    n_points=400
)

# Combine all points
X = np.vstack((cluster1, cluster2, cluster3))


# ============================================================
# Gaussian Mixture Clustering
# ============================================================

gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    random_state=42
)

gmm.fit(X)

labels = gmm.predict(X)


# ============================================================
# Plot Function
# ============================================================

def draw_ellipse(position, covariance, ax, color='red'):
    """
    Draw covariance ellipse.
    """

    if covariance.shape == (2, 2):

        eigenvalues, eigenvectors = np.linalg.eigh(covariance)

        order = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]

        angle = np.degrees(
            np.arctan2(eigenvectors[1, 0],
                       eigenvectors[0, 0])
        )

        width, height = 2 * np.sqrt(eigenvalues)

    else:
        width, height = 2 * np.sqrt(covariance)
        angle = 0

    ellipse = Ellipse(
        xy=position,
        width=width * 3,
        height=height * 3,
        angle=angle,
        edgecolor=color,
        fc='None',
        lw=2
    )

    ax.add_patch(ellipse)


# ============================================================
# Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    X[:, 0],
    X[:, 1],
    c=labels,
    cmap='viridis',
    s=15
)

# Draw learned ellipses
for mean, cov in zip(gmm.means_, gmm.covariances_):
    draw_ellipse(mean, cov, ax)

ax.set_title("Clustering of Fixed and Angular Elliptical Data")
ax.set_xlabel("X")
ax.set_ylabel("Y")

plt.grid(True)
plt.show()


# ============================================================
# Print Cluster Information
# ============================================================

print("\nCluster Centers:\n")
print(gmm.means_)

print("\nCovariance Matrices:\n")
print(gmm.covariances_)